# DuckDB exploration of the Parquet corpus

[DuckDB](https://duckdb.org/) is a zero-config columnar database that reads Parquet directly. Pointed at a `shard-*.parquet` glob, it gives you SQL over the corpus without any ETL.

The same queries shown below back the `corpus-prep explore` CLI command — this notebook is the long-form, narrated version.

## Setup

We assume you already produced a corpus under `data/corpus/` via `corpus-prep ingest data/raw -o data/corpus`. If not, run cell 2 below to generate a synthetic mini-corpus inline.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))

CORPUS_DIR = REPO_ROOT / "data" / "corpus"

In [ ]:
# If data/corpus/ is empty, build a synthetic corpus for the demo.
import json
import tempfile

if not list(CORPUS_DIR.glob("shard-*.parquet")):
    print("No shards under data/corpus/ — building a synthetic corpus...")
    from corpus_prep.pipeline import PipelineConfig, run_pipeline

    tmp = Path(tempfile.mkdtemp())
    in_dir = tmp / "in"
    in_dir.mkdir()

    samples = {
        "news.txt": (
            "Modelos de linguagem treinados em corpora de larga escala "
            "demonstram comportamentos emergentes a partir de certo numero "
            "de parametros. " * 3
        ),
        "recipe.md": (
            "# Baiao de dois\n\nPrato tradicional do sertao nordestino "
            "combinando arroz, feijao de corda, queijo coalho e carne seca. "
            "Cozinhar o feijao, refogar com cebola e alho. " * 3
        ),
        "meta.json": json.dumps(
            {
                "resumo": (
                    "Convocacao de candidatos para etapa de avaliacao de "
                    "titulos do processo seletivo simplificado. " * 3
                )
            }
        ),
    }
    for name, content in samples.items():
        (in_dir / name).write_text(content)

    CORPUS_DIR = tmp / "out"
    run_pipeline(
        PipelineConfig(input_dir=in_dir, output_dir=CORPUS_DIR, enable_filter=False)
    )
    print(f"Synthetic corpus at {CORPUS_DIR}")

GLOB = str(CORPUS_DIR / "shard-*.parquet")
GLOB

## 1. Connect DuckDB and create a view

DuckDB reads Parquet glob patterns directly — no ETL, no schema definition required.

In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE VIEW corpus AS SELECT * FROM '{GLOB}'")
print("Schema:")
print(con.execute("DESCRIBE corpus").fetchdf())

## 2. Basic counts

In [ ]:
con.execute("SELECT COUNT(*) AS total_documents FROM corpus").fetchdf()

In [ ]:
con.execute(
    """
    SELECT
        COUNT(*) AS docs,
        SUM(char_count) AS total_chars,
        ROUND(AVG(char_count), 1) AS avg_chars,
        MIN(char_count) AS min_chars,
        MAX(char_count) AS max_chars
    FROM corpus
    """
).fetchdf()

## 3. Composition by parser and language

In [ ]:
con.execute(
    """
    SELECT parser, COUNT(*) AS docs, SUM(char_count) AS total_chars
    FROM corpus
    GROUP BY parser
    ORDER BY docs DESC
    """
).fetchdf()

In [ ]:
con.execute(
    """
    SELECT COALESCE(language, '<unknown>') AS language, COUNT(*) AS docs
    FROM corpus
    GROUP BY language
    ORDER BY docs DESC
    """
).fetchdf()

## 4. Distribution of document lengths

A skewed distribution often reveals quality issues — too many very short documents may signal aggressive boilerplate stripping; very long ones often indicate boilerplate that survived.

In [ ]:
con.execute(
    """
    SELECT
        CASE
            WHEN char_count < 500 THEN '<500'
            WHEN char_count < 2000 THEN '500-2k'
            WHEN char_count < 10000 THEN '2k-10k'
            ELSE '>10k'
        END AS bucket,
        COUNT(*) AS docs
    FROM corpus
    GROUP BY bucket
    ORDER BY MIN(char_count)
    """
).fetchdf()

## 5. Search inside the corpus

Use SQL `LIKE` for ad-hoc string matching. For more sophisticated full-text search you'd want DuckDB's `fts` extension or a dedicated index.

In [ ]:
con.execute(
    """
    SELECT source_path, char_count, SUBSTR(text, 1, 80) AS preview
    FROM corpus
    WHERE LOWER(text) LIKE '%nordeste%' OR LOWER(text) LIKE '%sertao%'
    ORDER BY char_count DESC
    LIMIT 5
    """
).fetchdf()

## 6. Aggregating Document.metadata

The `metadata` column is a `MAP<string, string>` and DuckDB exposes it via `element_at()` (or `metadata['key']` on recent versions).

In [ ]:
# Example: how often pdf_native flagged needs_ocr.
con.execute(
    f"""
    SELECT
        element_at(metadata, 'needs_ocr')[1] AS needs_ocr,
        COUNT(*) AS docs
    FROM '{GLOB}'
    GROUP BY needs_ocr
    """
).fetchdf()

## Takeaways

- DuckDB + Parquet glob = zero-friction analytics over the corpus.
- The `metadata` map preserves parser-specific extras (e.g. `chars_per_page`, `needs_ocr`, `rows`/`columns`) without polluting the canonical schema.
- The same queries shipped here drive the `corpus-prep explore` CLI command.